# 07 · Your first API call
Level L2 · Part 2 · core · ~30 min · builds on 06

## Goal
Call Gemini with the official `google-genai` SDK, using a key read from `.env`, ask it to summarise one
Scout note, and print every field of the response that matters: the text, the model that answered,
the token usage and the stop reason.

## The idea
In lesson 06 the "model" was a Python function. A real model is the same function, but it runs on
someone else's server: you send it a request over HTTPS and get a response back.
To call it you need three things. An **API key** proves who you are and who pays. You keep it in an
**environment variable**, never in code. An **SDK client** turns a Python call into the HTTP request.
What comes back is a **response object**: the text, plus the facts you need to run a real system,
like how many tokens you paid for and why the model stopped.

Analogy: the API key is your library card, the client is the librarian, and the response is the
book plus a receipt.

## Picture
```mermaid
flowchart LR
    E[".env file"] -->|load_dotenv| V["os.environ"]
    V -->|api_key| C["genai.Client"]
    C -->|"HTTPS request"| G["Gemini on Vertex AI"]
    G -->|"JSON response"| R["response object"]
    R --> T["text"]
    R --> U["usage_metadata"]
    R --> F["finish_reason"]
    R --> M["model_version"]
```

## Already know this?
This is the first lesson of Part 2. If you can answer all of these, skip to lesson 18 (the Part's checkpoint).
1. Where should an API key live, and how does your code read it?
2. Does a model API remember your previous request? How does a chat keep its context?
3. Why stream a reply, and what is time-to-first-token?
4. How do you work out what one call cost, from the response alone?
5. Which errors should you retry, and how long should you wait between tries?

## Step by step
### 1. Start the recorder
Every model call in this course goes through `llm_replay`. The first time, with a key, it records the
real HTTP traffic to `../data/llm_cassettes/07_first-api-call.yaml`. After that it replays the
recording, so this notebook runs with no key, no cost and the same output. It prints which mode it is in.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("07_first-api-call")

LLM mode: replay (07_first-api-call.yaml)


### 2. Setup from lesson 06
Scout's notes, unchanged. We will ask Gemini to summarise one of them.

In [2]:
NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
print(len(NOTES), "notes")

6 notes


### 3. The key lives in `.env`, not in code
The topic folder has an `env.example` that lists the variables. You copy it to `.env` and paste your
key there. `.env` is listed in `.gitignore`, so it never reaches Git. Let's see what the example asks for,
printing only the names.

In [3]:
from pathlib import Path

example = Path("../env.example").read_text()
names = [line.split("=")[0].lstrip("# ") for line in example.splitlines() if "=" in line]
print(names)

['GOOGLE_CLOUD_API_KEY', 'TUT_LLM_MODEL', 'TUT_LLM_MODEL_LARGE', 'ANTHROPIC_API_KEY', 'OPENAI_API_KEY', 'TUT_LLM_LOCAL']


`load_dotenv` copies the lines of `.env` into `os.environ`, the process's **environment variables**.
It does not overwrite a variable that is already set in your shell. Never print the key itself. To check
it is there on your own machine, run `print(bool(os.environ.get(KEY_NAME)))`: `True` means found,
`False` means no `.env` (fine here, because the calls below replay from the recording).
We also pin the model id in one place and print it.

In [4]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
KEY_NAME = "GOOGLE_CLOUD_API_KEY"
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")   # pin the model id in one place
print("model:", MODEL)

model: gemini-3.5-flash-lite


### 4. Make one SDK client
`genai.Client` holds the key, the server address and the retry settings. `vertexai=True` with an
API key uses Gemini on Vertex AI (Google Cloud's "express mode"). Make one client and reuse it for
every call. When replaying with no key, a placeholder is fine: no request leaves your laptop.

In [5]:
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, api_key=os.environ.get(KEY_NAME) or "replay-no-key-needed")
print(type(client).__name__, "ready; Vertex AI:", client.vertexai)

Client ready; Vertex AI: True


One more setting. The SDK has an old "automatic function calling" feature that prints a warning on
plain calls. We switch it off; you will write tool calls by hand later in the course. We also cap the
reply at 60 tokens (lesson 05's max output).

In [6]:
CONFIG = types.GenerateContentConfig(
    max_output_tokens=60,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)
print(CONFIG.model_dump(exclude_none=True))

{'max_output_tokens': 60, 'automatic_function_calling': {'disable': True, 'maximum_remote_calls': 10}}


### 5. The smallest call
`client.models.generate_content` sends one request: a model id and the input text (`contents`).
`response.text` is the reply's text.

Why `generate_content`? Google also offers a newer Interactions API (`client.interactions.create`).
It is not available on Vertex AI with an API key (the server answers 404 Not Found), so this course
uses `generate_content`, which works on every Gemini setup. Lessons 09 and 10 say where the two differ.

In [7]:
hello = client.models.generate_content(model=MODEL, contents="Say hello in five words.", config=CONFIG)
print(hello.text)

Hello to you right now.


That was a real model on Google's servers (or its recording). Now the Scout version.

### 6. Ask Scout's question
We put note 5 into the prompt and ask for a one-sentence summary.

In [8]:
note = NOTES[4]
prompt = f"Summarise this note in one short sentence.\nNote {note['id']}: {note['text']}"
response = client.models.generate_content(model=MODEL, contents=prompt, config=CONFIG)
print(response.text)

The rover Perseverance successfully landed on Mars in February 2021.


### 7. Read the response, field by field
The text is only one part. The response also says which model actually answered, gives the request
an id, and tells you why the model stopped. All these come from the first (and here only)
**candidate**, one possible reply.

In [9]:
candidate = response.candidates[0]
print("text          :", response.text)
print("model_version :", response.model_version)
print("response_id   :", response.response_id)
print("create_time   :", response.create_time)
print("role          :", candidate.content.role)
print("finish_reason :", candidate.finish_reason.value)

text          : The rover Perseverance successfully landed on Mars in February 2021.
model_version : gemini-3.5-flash-lite
response_id   : XzrFaqS6GKvf-O4PmOLPmAg
create_time   : 2026-10-06 18:13:51.400676+00:00
role          : model
finish_reason : STOP


`finish_reason` is `STOP`: the model chose to end, like lesson 06's `end_turn`. `model_version` is the
exact model that served you. Log it, because it can differ from what you asked for.

### 8. The usage fields: what you paid for
`usage_metadata` counts tokens with the model's own tokenizer (lesson 03). Input is
`prompt_token_count`, output is `candidates_token_count`; `thoughts_token_count` is hidden
reasoning, billed as output, and is `None` when the model did not think.

In [10]:
usage = response.usage_metadata
thoughts = usage.thoughts_token_count or 0
print("input  (prompt_token_count)    :", usage.prompt_token_count)
print("output (candidates_token_count):", usage.candidates_token_count)
print("thinking (thoughts_token_count):", thoughts)
print("total  (total_token_count)     :", usage.total_token_count)
print("adds up:", usage.prompt_token_count + usage.candidates_token_count + thoughts == usage.total_token_count)

input  (prompt_token_count)    : 28
output (candidates_token_count): 15
thinking (thoughts_token_count): 0
total  (total_token_count)     : 43
adds up: True


The total is input plus output plus thinking. Lesson 12 turns these numbers into money.

### 9. Everything else in the response
The response is a Pydantic model, so `model_dump` turns it into plain Python. Here are all the fields
that have a value, two levels deep, plus the HTTP headers the server sent back.

In [11]:
data = response.model_dump(mode="json", exclude_none=True)
for key, value in data.items():
    if key == "sdk_http_response":
        print(f"{key}: headers -> {sorted(value['headers'])}")
    elif isinstance(value, list):
        print(f"{key}: list of {len(value)}, first has {sorted(value[0]) if value and isinstance(value[0], dict) else value}")
    elif isinstance(value, dict):
        print(f"{key}: {sorted(value)}")
    else:
        print(f"{key}: {value!r}")

sdk_http_response: headers -> ['content-length', 'content-type', 'date', 'server', 'transfer-encoding', 'vary', 'x-content-type-options', 'x-frame-options', 'x-xss-protection']
candidates: list of 1, first has ['content', 'finish_reason']
create_time: '2026-10-06T18:13:51.400676Z'
model_version: 'gemini-3.5-flash-lite'
response_id: 'XzrFaqS6GKvf-O4PmOLPmAg'
usage_metadata: ['candidates_token_count', 'candidates_tokens_details', 'prompt_token_count', 'prompt_tokens_details', 'total_token_count', 'traffic_type']


You have now met every field: the candidate (text and finish reason), the id, the time, the model
version and the usage, where `traffic_type` says how you were billed. No header carries your key:
`llm_replay` strips secrets before saving.

### 10. Same shape as Scout's fake model
Lesson 06's fake model returned `{"text", "input_tokens", "output_tokens", "stop_reason"}`. A small
adapter gives the real response that same shape, so code written for the fake model keeps working.

In [12]:
STOP_REASONS = {"STOP": "end_turn", "MAX_TOKENS": "max_tokens"}

def to_scout_reply(resp) -> dict:
    u = resp.usage_metadata
    reason = resp.candidates[0].finish_reason.value
    return {"text": resp.text, "input_tokens": u.prompt_token_count,
            "output_tokens": u.candidates_token_count + (u.thoughts_token_count or 0),
            "stop_reason": STOP_REASONS.get(reason, reason.lower())}

print(to_scout_reply(response))

{'text': 'The rover Perseverance successfully landed on Mars in February 2021.', 'input_tokens': 28, 'output_tokens': 15, 'stop_reason': 'end_turn'}


### 11. When the reply is cut off
Give the model only 10 output tokens and it stops mid-sentence. The text still looks like an answer,
so only `finish_reason` tells you it is incomplete.

In [13]:
short = types.GenerateContentConfig(max_output_tokens=10,
                                    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
cut = client.models.generate_content(model=MODEL, contents=prompt, config=short)
print(to_scout_reply(cut))

{'text': 'The rover Perseverance landed on Mars', 'input_tokens': 28, 'output_tokens': 6, 'stop_reason': 'max_tokens'}


`max_tokens`: the same stop condition you built by hand in lesson 02. Notice the output count is
below 10: the count alone cannot tell you the reply was cut, `finish_reason` can. Lesson 13 handles every
finish reason, including safety blocks and empty replies.

## At work
- **Keys come from the environment, never from code.** Teams use `.env` (ignored by Git) on laptops, a
  secret manager in production, separate keys per environment, and rotate a key the day it leaks.
  Logs must never include auth headers. → lesson 50
- **Every response's usage is logged.** Model, tokens in/out, thinking tokens, stop reason and latency
  go into a log line per call; that is how teams track cost and catch runaway prompts. → lessons 12 and 49
- **The model id is pinned in config, and `model_version` is logged.** Vendors retire models on a
  schedule (Gemini 2.5 is already restricted for new users), so the id lives in one place. → lessons 17 and 53
- **One client per process, reused.** The client holds connections and retry settings; SDKs retry some
  errors automatically, and knowing which ones matters. → lesson 14

## What just happened
- `load_dotenv` put the key from `.env` into `os.environ`; we checked it was set without printing it.
- One `genai.Client` sent HTTPS requests to Gemini on Vertex AI; `llm_replay` recorded them so the notebook replays with no key.
- `generate_content` returned a response object: the summary in `.text`, the serving model in `model_version`, an id, and `finish_reason` `STOP`.
- `usage_metadata` gave input, output and thinking tokens, and they add up to the total.
- `to_scout_reply` turned the response into lesson 06's reply shape.
- With 10 output tokens the reply was cut, and only `finish_reason` `MAX_TOKENS` said so.

## Common mistakes
**1. Reading another vendor's field names.** Anthropic and OpenAI call it `response.usage`. Gemini does not.

In [14]:
print(response.usage.input_tokens)

AttributeError: 'GenerateContentResponse' object has no attribute 'usage'

Fix: Gemini's fields are `usage_metadata.prompt_token_count` and `candidates_token_count`. Lesson 10
compares all three vendors side by side.

In [15]:
print(response.usage_metadata.prompt_token_count, response.usage_metadata.candidates_token_count)

28 15


**2. Trusting `.text` without checking why the model stopped.** The cut-off reply is a plausible
fragment. Saving it as Scout's summary would store half a sentence. A check makes the problem loud:

In [16]:
def summary_text(resp) -> str:
    reason = resp.candidates[0].finish_reason.value
    assert reason == "STOP", f"reply incomplete ({reason}): {resp.text!r}"
    return resp.text

print(summary_text(response))
print(summary_text(cut))

The rover Perseverance successfully landed on Mars in February 2021.


AssertionError: reply incomplete (MAX_TOKENS): 'The rover Perseverance landed on Mars'

Fix: always read `finish_reason` before using the text, and raise `max_output_tokens` (or retry) when it
is `MAX_TOKENS`. The first call above passed the check and printed the summary; only the cut one failed.

## Try it
Write `describe(resp)` that returns one line like
`gemini-3.5-flash-lite | STOP | 28 in, 15 out, 0 thinking | ok`, ending in `CUT` instead of `ok` when the
reply did not finish normally (your numbers may differ by a token or two). Run it on `hello`, `response`
and `cut`. No new model calls needed.

In [17]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [18]:
llm_replay.stop()

## Key terms
- **API key** — a secret string that identifies you (and who pays) to a model provider; kept in an environment variable, never in code.
- **environment variable** — a named value the operating system gives a running process; read in Python with `os.environ`; `load_dotenv` fills it from a `.env` file.
- **SDK client** — the vendor library object (here `genai.Client`) that holds your key and settings and turns Python calls into HTTP requests.
- **response object** — what an API call returns: the reply text plus metadata such as model version, id, finish reason and usage.
- **candidate** — one possible reply inside a Gemini response; by default there is exactly one.
- **finish reason** — why the model stopped writing (`STOP`, `MAX_TOKENS`, ...); Gemini's name for the stop reason.
- **usage fields** — the token counts a response reports (input, output, thinking, total); the basis of cost.